In [ ]:
# Authentication & Google Drive-free version of the below cells, uncomment if there are problems
# COLAB ONLY CELLS
try:
   import google.colab
   IN_COLAB = True
   !pip3 install transformers  # https://huggingface.co/docs/transformers/installation
#    !nvidia-smi                 # Check which GPU has been chosen for us
   !rm -rf logs
   # Download the dataset from personal drive
   !mkdir data
   !wget --no-check-certificate 'https://docs.google.com/uc?export=download&id=19jcMX4KFwVAp4yvgvw1GXSnSgpoQytqg' -O data/training_set.json
except:
   IN_COLAB = False

mkdir: cannot create directory ‘data’: File exists
--2022-01-21 14:27:28--  https://docs.google.com/uc?export=download&id=19jcMX4KFwVAp4yvgvw1GXSnSgpoQytqg
Resolving docs.google.com (docs.google.com)... 74.125.129.102, 74.125.129.113, 74.125.129.100, ...
Connecting to docs.google.com (docs.google.com)|74.125.129.102|:443... connected.
HTTP request sent, awaiting response... 302 Moved Temporarily
Location: https://doc-0o-b4-docs.googleusercontent.com/docs/securesc/ha0ro937gcuc7l7deffksulhg5h7mbp1/ri61gh4jas3lnqlaur2h9vj5hnidt00s/1642775175000/11026221962362482708/*/19jcMX4KFwVAp4yvgvw1GXSnSgpoQytqg?e=download [following]
--2022-01-21 14:27:29--  https://doc-0o-b4-docs.googleusercontent.com/docs/securesc/ha0ro937gcuc7l7deffksulhg5h7mbp1/ri61gh4jas3lnqlaur2h9vj5hnidt00s/1642775175000/11026221962362482708/*/19jcMX4KFwVAp4yvgvw1GXSnSgpoQytqg?e=download
Resolving doc-0o-b4-docs.googleusercontent.com (doc-0o-b4-docs.googleusercontent.com)... 142.250.152.132, 2607:f8b0:4001:c56::84
Connecting 

In [ ]:
%load_ext tensorboard

import os
import requests
import zipfile
from tqdm import tqdm
import time
import random
import datetime
from IPython.display import display
from functools import partial
from itertools import product
from typing import List, Dict, Callable, Sequence, Tuple


import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

%matplotlib inline

# Fix random seed for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

############################## CONFIG ###########################

SMALL_TRAIN_LEN = 20 # NUMBER OF ARTICLES
SMALL_VAL_LEN = 5
BATCH_SIZE = 16 # NUMBER OF PARAGRAPH + QUESTION PAIRS
VAL_BATCH_SIZE = 4

In [ ]:
resolver = tf.distribute.cluster_resolver.TPUClusterResolver(tpu='')
tf.config.experimental_connect_to_cluster(resolver)
# This is the TPU initialization code that has to be at the beginning.
tf.tpu.experimental.initialize_tpu_system(resolver)
print("All devices: ", tf.config.list_logical_devices('TPU'))


INFO:tensorflow:Deallocate tpu buffers before initializing tpu system.


INFO:tensorflow:Deallocate tpu buffers before initializing tpu system.


INFO:tensorflow:Initializing the TPU system: grpc://10.126.215.18:8470


INFO:tensorflow:Initializing the TPU system: grpc://10.126.215.18:8470


INFO:tensorflow:Finished initializing TPU system.


INFO:tensorflow:Finished initializing TPU system.


All devices:  [LogicalDevice(name='/job:worker/replica:0/task:0/device:TPU:0', device_type='TPU'), LogicalDevice(name='/job:worker/replica:0/task:0/device:TPU:1', device_type='TPU'), LogicalDevice(name='/job:worker/replica:0/task:0/device:TPU:2', device_type='TPU'), LogicalDevice(name='/job:worker/replica:0/task:0/device:TPU:3', device_type='TPU'), LogicalDevice(name='/job:worker/replica:0/task:0/device:TPU:4', device_type='TPU'), LogicalDevice(name='/job:worker/replica:0/task:0/device:TPU:5', device_type='TPU'), LogicalDevice(name='/job:worker/replica:0/task:0/device:TPU:6', device_type='TPU'), LogicalDevice(name='/job:worker/replica:0/task:0/device:TPU:7', device_type='TPU')]


In [ ]:
strategy = tf.distribute.TPUStrategy(resolver)


INFO:tensorflow:Found TPU system:


INFO:tensorflow:Found TPU system:


INFO:tensorflow:*** Num TPU Cores: 8


INFO:tensorflow:*** Num TPU Cores: 8


INFO:tensorflow:*** Num TPU Workers: 1


...

INFO:tensorflow:*** Available Device: _DeviceAttributes(/job:worker/replica:0/task:0/device:TPU:7, TPU, 0, 0)


INFO:tensorflow:*** Available Device: _DeviceAttributes(/job:worker/replica:0/task:0/device:TPU_SYSTEM:0, TPU_SYSTEM, 0, 0)


INFO:tensorflow:*** Available Device: _DeviceAttributes(/job:worker/replica:0/task:0/device:TPU_SYSTEM:0, TPU_SYSTEM, 0, 0)


INFO:tensorflow:*** Available Device: _DeviceAttributes(/job:worker/replica:0/task:0/device:XLA_CPU:0, XLA_CPU, 0, 0)


INFO:tensorflow:*** Available Device: _DeviceAttributes(/job:worker/replica:0/task:0/device:XLA_CPU:0, XLA_CPU, 0, 0)


In [ ]:
TRAINING_FILE = os.path.join('data', 'training_set.json')
with open(TRAINING_FILE, 'r') as f:
    questions = json.load(f)

In [ ]:
TRAIN_SPLIT_ELEM = int(len(questions['data']) / 100 * 75)
print("Using {} articles for validation and {} for training".format(
    len(questions['data']) - TRAIN_SPLIT_ELEM, TRAIN_SPLIT_ELEM)
)

data = random.sample(questions['data'], len(questions['data'])) # reshuffle the samples

Using 111 articles for validation and 331 for training


In [ ]:
train_dataset = {'data': data[:TRAIN_SPLIT_ELEM]} # recreate the original dataset structure lost by shuffling through the dictionary
val_dataset = {'data': data[TRAIN_SPLIT_ELEM:]}

# we also create a small training set to test the model while building it, just to speed up

small_data = random.sample(train_dataset["data"], SMALL_TRAIN_LEN)
small_train_dataset = {'data': small_data}
small_val_data = random.sample(val_dataset["data"], SMALL_VAL_LEN)
small_val_dataset = {'data': small_val_data}

## Question analysis
Lets check for type of question which two word precede the answer

In [53]:
import re
import string


def get_2_previous_word(text, answer):
    re_pattern = r'\S+\s\S+\s"?(?='+answer_text+')'
    try: 
        x = re.search(re_pattern, context)           
        # print(x.group())
        match = x.group()
        match = match.translate(str.maketrans('','',string.punctuation))
        match = match.strip().lower()
        return match
        # ws_dict[w][x.group()]+=1
        # print(ws_dict[w][x.group()])
    except Exception as e:
        return ""
ws = ["Who", "What", "Why", "How", "When", "Where", "Whose", "Which", "Others"]
# ws_dict = dict.fromkeys(ws, "")
ws_dict = {key:dict() for key in ws}

for article in train_dataset['data']:
    for paragraph in article['paragraphs']:
        context = paragraph['context']
        for q in paragraph['qas']:
            answer = q['answers'][0]
            answer_start = answer['answer_start']
            answer_text = answer['text']
            question = q['question']
            is_others = True
            for w in ws_dict.keys():
                if w in question:
                    previous_text = get_2_previous_word(context, answer_text)
                    # print(w)
                    if previous_text:
                        if previous_text in ws_dict[w]:
                            ws_dict[w][previous_text]+=1
                        else:
                            ws_dict[w][previous_text]=1
                        # print(e)
                    is_others = False
                    break
            if is_others:
                previous_text = get_2_previous_word(context, answer_text)
                if previous_text:
                    if previous_text in ws_dict["Others"]:
                        ws_dict["Others"][previous_text]+=1
                    else:
                        ws_dict["Others"][previous_text]=1



In [55]:
k = 15
for w in ws:
    df = pd.DataFrame(ws_dict[w].items(), columns=["key", "count"])
    df.sort_values(by='count', inplace=True, ascending=False)
    print("Top Previous word of {} questions".format(w))
    display(df.head(k))

Top Previous word of Who questions


,key,count
11,by the,114
10,of the,57
9,led by,47
94,such as,42
591,designed by,24
79,according to,24
80,to the,24
25,developed by,23
233,with the,20
198,written by,18


Top Previous word of What questions


,key,count
39,of the,448
116,such as,316
14,known as,230
84,as the,229
231,in the,201
229,by the,174
99,to the,166
284,and the,126
82,is the,120
1046,with the,89


Top Previous word of Why questions


...

,key,count


Top Previous word of Which questions


,key,count
75,of the,67
21,in the,50
104,such as,47
1,by the,35
218,to the,32
36,and the,27
178,as the,22
256,with the,21
271,is the,15
73,on the,14


Top Previous word of Others questions


,key,count
20,in the,380
5,of the,338
202,to the,108
198,such as,97
239,from the,80
3,and the,73
356,known as,67
38,during the,66
312,by the,65
467,at the,64


In [ ]:
from transformers import DistilBertTokenizerFast, TFDistilBertModel, BertTokenizerFast, TFBertModel
# We are using a tokenizer that derives from a "normal" (and not "large") BERT model
# moreover, it ignores casing (uncased)
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased') # TODO: cased or uncased?
tokenizerB = BertTokenizerFast.from_pretrained('bert-base-uncased')

In [ ]:
MAX_LEN_PAIRS = 512

def find_start_end_token_one_hot_encoded(answers: Dict, offsets: List[Tuple[int]], tokenizer, token_type_ids=False) -> int:
    '''
    This function returns the starting and ending token of the answer, already one hot encoded and ready for binary crossentropy
    Inputs:
        answers: List[Dict] --> for each question, a list of answers. Each answer contains:
            - answer_start: the index of the starting character
            - text: the text of the answer, that we exploit through the number of chars that it containts
        offsets: List[Tuple[int]] --> the tokenizer from HuggingFace transforms the sentence (question+context)
            into a sequence of tokens. Offsets keeps track of the character start and end indexes for each token
    Output:
        result: Dict --> each key contains only one array, the one-hot encoded version of, respectively, the start
            and end token of the answer in the sentence (question+context)
    '''
    result = {
        "out_S": np.zeros(len(offsets)),
        "out_E": np.zeros(len(offsets))
    }   

    for answer in answers:
        starting_char = answer['answer_start']
        answer_len = len(answer['text'])

        for i in range(1, len(offsets)): # we skip the first token, [CLS], that has (0,0) as a tuple
            # We cycle through all the tokens of the question, until we find (0,0), which determines the separator
            if offsets[i] == (0,0): # The [SEP] special char --> this indicates the beginning of the context
                for j in range(1, len(offsets)-i-1): # We skip the first and the last tokens, both special tokens
                    # If the starting char is in the interval, the index (j) of its position inside the context, 
                    # plus the length of the question (i) is the right index
                    if (starting_char >= offsets[i+j][0]) and (starting_char <= offsets[i+j][1]):
                        result["out_S"][i+j] += 1
                    # if the ending char (starting + length -1) is in the interval, same as above
                    if (starting_char + answer_len - 1 >= offsets[i+j][0]) and (starting_char + answer_len - 1 < offsets[i+j][1]):
                        result["out_E"][i+j] += 1
                        break
                # After this cycle, we must check other answers
                break
    
    return result

def create_data_for_dataset(data, tokenizer, max_pairs_length=512, token_type_ids=False):
    '''
    This function takes in input the whole data structure and iteratively composes question+context pairs, plus their label
    Inputs:
        data: Dict --> the data structure containing the data
    Outputs:
        tf.data.Dataset --> the data structure containing (features, labels) that will be fed to the model during fitting
        more specifically:
        features: Dict --> keys:
            - input_ids: array of token ids
            - attention_mask: array indicating if the corresponding token is padding or not
        labels: Dict --> keys:
            - gt_S: array representing the index of the initial token of the answer, one-hot encoded
            - gt_E: array representing the index of the final token of the answer, one-hot encoded

    This function, for each article in "data", extracts all paragraphs (and their text, the "context"), for each paragraph, all questions_and_answers
    At this point, it tokenizes (question+context) while truncating and padding up to MAX_LEN_PAIRS
    Moreover, it also returns the "attention_mask", an array that tells if the token is padding or normal, that will be used by the model

    It also keeps track, through "find_start_end_token_one_hot_encoded", of the index of the initial and final token of the answer, the labels for the model

    In the end, it returns a tf.data.Dataset with the structure (features, labels), to be injected directly in the fit method of the model
    '''
    features = []
    labels = []

    for article in tqdm(data["data"]):
        for paragraph in article["paragraphs"]:
            for question_and_answer in paragraph["qas"]:
                ### QUESTION AND CONTEXT TOKENIZATION ###
                # For question answering with BERT we need to encode both 
                # question and context, and this is the way in which 
                # HuggingFace's BertTokenizer does it.
                # The tokenizer returns a dictionary containing all the information we need
                encoded_inputs = tokenizer(
                    question_and_answer["question"],    # First we pass the question
                    paragraph["context"],               # Then the context
                    max_length =  max_pairs_length,         # We want to pad and truncate to this length
                    truncation = True,
                    padding = 'max_length',             # Pads all sequences to 512.
                                                        # If "True" it would pad to the longest sentence in the batch 
                                                        # (in this case we only use 1 sentence, so no padding at all)
                    # return_token_type_ids = True,     # IF USING BERT, DistilBert does not need it 
                    return_token_type_ids = token_type_ids,      # Return if the token is from sentence 0 or sentence 1 
                    return_attention_mask = True,       # Return if it's a pad token or not
                    return_offsets_mapping = True       # Really important --> returns each token's first and last char position in the original sentence 
                )
                
                ### MAPPING OF THE START OF THE ANSWER BETWEEN CHARS AND TOKENS ###
                # We want to pass from the starting position in chars to the starting position in tokens
                label = find_start_end_token_one_hot_encoded(
                    # We pass the list of answers (usually there is still one per question,
                    #   but we mustn't assume anything)
                    answers = question_and_answer["answers"],
                    # And also the inputs offset mapping just recieved from the tokenizer
                    offsets = encoded_inputs["offset_mapping"],
                    tokenizer = tokenizer
                )
                
                encoded_inputs.pop("offset_mapping", None) # Removes the offset mapping, not useful anymore 
                                                           # ("None" is used because otherwise KeyError could be raised if the key wasn't present)
                features.append(encoded_inputs)
                labels.append(label)

                # DO NOT KNOW IF IT IS NEEDED
                '''
                ### ANSWER TOKENIZATION ###
                # use the same tokenizer also to tokenize the answers
                encoded_answer = tokenizer(
                    question_and_answer["answers"][0]["text"],  # here we only need to pass the answer
                    max_length=MAX_LEN_ANSWERS,
                    truncation = True,
                    padding = 'max_length',
                    add_special_tokens = False,                 # the answer will only be used for the loss, not as input to the model, it does not need special tokens [CLS] and [SEP]
                    return_token_type_ids = False,              # only one sentence
                    return_attention_mask = True)               # still interested in padding tokens
                
                # we add to the dictionary of the pair question-context the token ids of the answer and its mask
                encoded_inputs["answer_ids"] = encoded_answer["input_ids"]
                encoded_inputs["answer_mask"] = encoded_answer["attention_mask"]
                '''

    print("Creating dataset")
    return tf.data.Dataset.from_tensor_slices((
        pd.DataFrame.from_dict(features).to_dict(orient="list"),  # dataframe for features 
        pd.DataFrame.from_dict(labels).to_dict(orient="list")                                                    # dataframe for labels 
    ))

In [ ]:
# OSS: remember to call fit with this structure {"input_ids": train_df["input_ids"], ...}
# Load the model pretrained on the masked input + sentence completion task
with strategy.scope():
  transformer_model = TFDistilBertModel.from_pretrained("distilbert-base-uncased", output_hidden_states = True) # add here config for the model, ex. also attention outputs
  transformer_modelB = TFBertModel.from_pretrained("bert-base-uncased", output_hidden_states = True)

Some layers from the model checkpoint at distilbert-base-uncased were not used when initializing TFDistilBertModel: ['vocab_layer_norm', 'vocab_projector', 'activation_13', 'vocab_transform']
- This IS expected if you are initializing TFDistilBertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFDistilBertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
All the layers of TFDistilBertModel were initialized from the model checkpoint at distilbert-base-uncased.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFDistilBertModel for predictions without further training.
Some layers from the model checkpoint at bert-base-uncased were not used

In [ ]:
def create_model(transformer_model, max_pairs_length=512, bert=False):
    input_ids = tf.keras.Input(shape=(max_pairs_length, ), name="input_ids", dtype='int32')
    attention_mask = tf.keras.Input(shape=(max_pairs_length, ), name="attention_mask", dtype='int32')
    token_type_ids = tf.keras.Input(shape=(max_pairs_length, ), name="token_type_ids", dtype='int32') # uncomment if using BERT
    if not bert:
      transformer = transformer_model(
          {
              "input_ids": input_ids,
              "attention_mask": attention_mask,
              # "token_type_ids": token_type_ids # uncomment if using BERT
          }
      )
    else:
      transformer = transformer_model(
          {
              "input_ids": input_ids,
              "attention_mask": attention_mask,
              "token_type_ids": token_type_ids # uncomment if using BERT
          }
      )
    # # TODO: chose which layers
    # hidden_states = transformer.hidden_states
    # chosen_states_idx = [3, 4, 5, 6]

    # # TODO: chose merging method
    # chosen_hidden_states = layers.concatenate(
    #     tuple([hidden_states[i] for i in chosen_states_idx])
    # )
    chosen_hidden_states = transformer.last_hidden_state
    # output = layers.Bidirectional(layers.LSTM(64, return_sequences = True, activation = "relu"))(chosen_hidden_states)
    # output = layers.Dense(2, activation = "softmax")(output) # 2 because we need both 

    out_S = layers.Dense(1)(chosen_hidden_states) # dot product between token representation and start vector
    out_S = layers.Flatten()(out_S)
    out_S = layers.Softmax(name="out_S")(out_S)

    out_E = layers.Dense(1)(chosen_hidden_states) # dot product between token representation and end vector
    out_E = layers.Flatten()(out_E)
    out_E = layers.Softmax(name="out_E")(out_E)
    
  
    if bert:
      model = tf.keras.models.Model(
          inputs=[input_ids, attention_mask, token_type_ids],
          outputs = [out_S, out_E]
      )
    else:
        model = tf.keras.models.Model(
          inputs=[input_ids, attention_mask],
          outputs = [out_S, out_E]
      )
    return model


### Ensemble

#### DistillBert

First lets train using Distillbert as transformer model.

Lets process the train set and the validation set using the distill bert tokenizer.

In [ ]:
batch_size = 32
max_pairs_length = 400
train_ds = create_data_for_dataset(train_dataset, tokenizer, max_pairs_length).batch(batch_size)
val_ds = create_data_for_dataset(val_dataset, tokenizerB, max_pairs_length).batch(batch_size)

100%|██████████| 331/331 [00:58<00:00,  5.70it/s]


Creating dataset


100%|██████████| 111/111 [00:22<00:00,  5.01it/s]


Creating dataset


In [ ]:
with strategy.scope():
  model = create_model(transformer_model, max_pairs_length)

model.compile(tf.keras.optimizers.Adam(3e-5), 
        loss={'out_S': 'binary_crossentropy', 'out_E': 'binary_crossentropy'},
        metrics={'out_S': 'accuracy', 'out_E':'accuracy'})

model.summary()

Model: "model_5"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 attention_mask (InputLayer)    [(None, 400)]        0           []                               
                                                                                                  
 input_ids (InputLayer)         [(None, 400)]        0           []                               
                                                                                                  
 tf_distil_bert_model (TFDistil  TFBaseModelOutput(l  66362880   ['attention_mask[0][0]',         
 BertModel)                     ast_hidden_state=(N               'input_ids[0][0]']              
                                one, 400, 768),                                                   
                                 hidden_states=((No                                         

In [ ]:

model.fit(
        train_ds, 
        validation_data=val_ds,
        epochs=3, 
        # callbacks=[cp_callback]
        )

Epoch 1/3


INFO:absl:TPU has inputs with dynamic shapes: [<tf.Tensor 'Const:0' shape=() dtype=int32>, <tf.Tensor 'cond/Identity:0' shape=(None, 400) dtype=int32>, <tf.Tensor 'cond/Identity_8:0' shape=(None, 400) dtype=int32>, <tf.Tensor 'cond/Identity_16:0' shape=(None, 400) dtype=float64>, <tf.Tensor 'cond/Identity_24:0' shape=(None, 400) dtype=float64>]
INFO:absl:TPU has inputs with dynamic shapes: [<tf.Tensor 'Const:0' shape=() dtype=int32>, <tf.Tensor 'cond/Identity:0' shape=(None, 400) dtype=int32>, <tf.Tensor 'cond/Identity_8:0' shape=(None, 400) dtype=int32>, <tf.Tensor 'cond/Identity_16:0' shape=(None, 400) dtype=float64>, <tf.Tensor 'cond/Identity_24:0' shape=(None, 400) dtype=float64>]


2034/2034 [==============================] - ETA: 0s - loss: 0.0119 - out_S_loss: 0.0062 - out_E_loss: 0.0056 - out_S_accuracy: 0.5347 - out_E_accuracy: 0.5807

INFO:absl:TPU has inputs with dynamic shapes: [<tf.Tensor 'Const:0' shape=() dtype=int32>, <tf.Tensor 'cond/Identity:0' shape=(None, 400) dtype=int32>, <tf.Tensor 'cond/Identity_8:0' shape=(None, 400) dtype=int32>, <tf.Tensor 'cond/Identity_16:0' shape=(None, 400) dtype=float64>, <tf.Tensor 'cond/Identity_24:0' shape=(None, 400) dtype=float64>]


2034/2034 [==============================] - 296s 118ms/step - loss: 0.0119 - out_S_loss: 0.0062 - out_E_loss: 0.0056 - out_S_accuracy: 0.5347 - out_E_accuracy: 0.5807 - val_loss: 0.0095 - val_out_S_loss: 0.0051 - val_out_E_loss: 0.0044 - val_out_S_accuracy: 0.6308 - val_out_E_accuracy: 0.6786
Epoch 2/3
2034/2034 [==============================] - 196s 96ms/step - loss: 0.0079 - out_S_loss: 0.0042 - out_E_loss: 0.0036 - out_S_accuracy: 0.6776 - out_E_accuracy: 0.7258 - val_loss: 0.0093 - val_out_S_loss: 0.0050 - val_out_E_loss: 0.0043 - val_out_S_accuracy: 0.6429 - val_out_E_accuracy: 0.6904
Epoch 3/3
2034/2034 [==============================] - 197s 97ms/step - loss: 0.0060 - out_S_loss: 0.0033 - out_E_loss: 0.0028 - out_S_accuracy: 0.7460 - out_E_accuracy: 0.7917 - val_loss: 0.0099 - val_out_S_loss: 0.0053 - val_out_E_loss: 0.0046 - val_out_S_accuracy: 0.6463 - val_out_E_accuracy: 0.6915


Lets save the checkpoint from the training and the weights

In [ ]:
checkpoint_path = "training_1/cp.ckpt"
checkpoint = tf.train.Checkpoint(model=model)
local_device_option = tf.train.CheckpointOptions(experimental_io_device="/job:localhost")
checkpoint.write(checkpoint_path, options=local_device_option)

'training_1/cp.ckpt'

In [ ]:
model.save_weights("training/weights_distill.h5")

In [ ]:
model.load_weights("training/weights_distill.h5")

#### Bert
Now we repeat the process with Bert.

In [ ]:
train_dsB = create_data_for_dataset(train_dataset, tokenizerB, max_pairs_length, True).batch(batch_size)
val_dsB = create_data_for_dataset(val_dataset, tokenizerB, max_pairs_length, True).batch(batch_size)

100%|██████████| 331/331 [00:54<00:00,  6.12it/s]


Creating dataset


100%|██████████| 111/111 [00:17<00:00,  6.43it/s]


Creating dataset


In [ ]:
with strategy.scope():
  modelB = create_model(transformer_modelB, max_pairs_length, True)
modelB.compile(tf.keras.optimizers.Adam(3e-5), 
        loss=tf.losses.BinaryCrossentropy(from_logits=False),
        metrics={'out_S': 'accuracy', 'out_E':'accuracy'})

modelB.summary()

Model: "model_4"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 attention_mask (InputLayer)    [(None, 400)]        0           []                               
                                                                                                  
 input_ids (InputLayer)         [(None, 400)]        0           []                               
                                                                                                  
 token_type_ids (InputLayer)    [(None, 400)]        0           []                               
                                                                                                  
 tf_bert_model (TFBertModel)    TFBaseModelOutputWi  109482240   ['attention_mask[0][0]',         
                                thPoolingAndCrossAt               'input_ids[0][0]',        

In [ ]:

modelB.fit(
        train_dsB, 
        validation_data=val_dsB,
        epochs=3, 
        # callbacks=[cp_callback]
        )

Epoch 1/3


INFO:absl:TPU has inputs with dynamic shapes: [<tf.Tensor 'Const:0' shape=() dtype=int32>, <tf.Tensor 'cond/Identity:0' shape=(None, 400) dtype=int32>, <tf.Tensor 'cond/Identity_8:0' shape=(None, 400) dtype=int32>, <tf.Tensor 'cond/Identity_16:0' shape=(None, 400) dtype=int32>, <tf.Tensor 'cond/Identity_24:0' shape=(None, 400) dtype=float64>, <tf.Tensor 'cond/Identity_32:0' shape=(None, 400) dtype=float64>]


INFO:absl:TPU has inputs with dynamic shapes: [<tf.Tensor 'Const:0' shape=() dtype=int32>, <tf.Tensor 'cond/Identity:0' shape=(None, 400) dtype=int32>, <tf.Tensor 'cond/Identity_8:0' shape=(None, 400) dtype=int32>, <tf.Tensor 'cond/Identity_16:0' shape=(None, 400) dtype=int32>, <tf.Tensor 'cond/Identity_24:0' shape=(None, 400) dtype=float64>, <tf.Tensor 'cond/Identity_32:0' shape=(None, 400) dtype=float64>]


2034/2034 [==============================] - ETA: 0s - loss: 0.0105 - out_S_loss: 0.0055 - out_E_loss: 0.0050 - out_S_accuracy: 0.5863 - out_E_accuracy: 0.6346

INFO:absl:TPU has inputs with dynamic shapes: [<tf.Tensor 'Const:0' shape=() dtype=int32>, <tf.Tensor 'cond/Identity:0' shape=(None, 400) dtype=int32>, <tf.Tensor 'cond/Identity_8:0' shape=(None, 400) dtype=int32>, <tf.Tensor 'cond/Identity_16:0' shape=(None, 400) dtype=int32>, <tf.Tensor 'cond/Identity_24:0' shape=(None, 400) dtype=float64>, <tf.Tensor 'cond/Identity_32:0' shape=(None, 400) dtype=float64>]


2034/2034 [==============================] - 500s 201ms/step - loss: 0.0105 - out_S_loss: 0.0055 - out_E_loss: 0.0050 - out_S_accuracy: 0.5863 - out_E_accuracy: 0.6346 - val_loss: 0.0088 - val_out_S_loss: 0.0047 - val_out_E_loss: 0.0041 - val_out_S_accuracy: 0.6617 - val_out_E_accuracy: 0.7118
Epoch 2/3
2034/2034 [==============================] - 340s 167ms/step - loss: 0.0069 - out_S_loss: 0.0037 - out_E_loss: 0.0032 - out_S_accuracy: 0.7160 - out_E_accuracy: 0.7664 - val_loss: 0.0089 - val_out_S_loss: 0.0048 - val_out_E_loss: 0.0041 - val_out_S_accuracy: 0.6690 - val_out_E_accuracy: 0.7179
Epoch 3/3
2034/2034 [==============================] - 342s 168ms/step - loss: 0.0051 - out_S_loss: 0.0028 - out_E_loss: 0.0023 - out_S_accuracy: 0.7808 - out_E_accuracy: 0.8311 - val_loss: 0.0099 - val_out_S_loss: 0.0052 - val_out_E_loss: 0.0047 - val_out_S_accuracy: 0.6652 - val_out_E_accuracy: 0.7170


In [ ]:
checkpoint_path = "training_1/cp.ckpt"
checkpoint = tf.train.Checkpoint(model=modelB)
local_device_option = tf.train.CheckpointOptions(experimental_io_device="/job:localhost")
checkpoint.write(checkpoint_path, options=local_device_option)

'training_1/cp.ckpt'

In [ ]:
modelB.save_weights("training/weights_bert.h5")

In [ ]:
modelB.load_weights("/training/weights_bert.h5")

In [ ]:
def start_end_token_from_probabilities(pstartv: np.array, 
                                       pendv: np.array, 
                                       dim:int=512) -> List[List[int]]:
    '''
    Returns a List of [StartToken, EndToken] elements computed from the batch outputs.
    '''
    idxs = []
    for i in range(pstartv.shape[0]):
        pstart = np.stack([pstartv[i,:]]*dim, axis=1)
        pend = np.stack([pendv[i,:]]*dim, axis=0)
        sums = pstart + pend
        sums = np.triu(sums, k=1) # Zero out lower triangular matrix + diagonal
        val = np.argmax(sums)
        row = val // dim
        col = val - dim*row
        idxs.append([row,col])
    return idxs

In [ ]:
import string
import re

def normalize_answer(s):
  """Lower text and remove punctuation, articles and extra whitespace."""
  def remove_articles(text):
    regex = re.compile(r'\b(a|an|the)\b', re.UNICODE)
    return re.sub(regex, ' ', text)
  def white_space_fix(text):
    return ' '.join(text.split())
  def remove_punc(text):
    exclude = set(string.punctuation)
    return ''.join(ch for ch in text if ch not in exclude)
  def lower(text):
    return text.lower()
  return white_space_fix(remove_articles(remove_punc(lower(s))))

def get_tokens(s):
  if not s: return []
  return normalize_answer(s).split()

In [ ]:
import collections

def compute_exact(a_gold, a_pred):
  return int(normalize_answer(a_gold) == normalize_answer(a_pred))

def compute_f1(a_gold, a_pred):
  gold_toks = get_tokens(a_gold)
  pred_toks = get_tokens(a_pred)
  common = collections.Counter(gold_toks) & collections.Counter(pred_toks)
  num_same = sum(common.values())
  if len(gold_toks) == 0 or len(pred_toks) == 0:
    # If either is no-answer, then F1 is 1 if they agree, 0 otherwise
    return int(gold_toks == pred_toks)
  if num_same == 0:
    return 0
  precision = 1.0 * num_same / len(pred_toks)
  recall = 1.0 * num_same / len(gold_toks)
  f1 = (2 * precision * recall) / (precision + recall)
  return f1

In [ ]:
dataset_ds = train_ds = create_data_for_dataset(small_val_dataset, max_pairs_length)

In [ ]:
scores = {"bert":{"f1":[],"em":[]}, "distillbert": {"f1":[],"em":[]}, "ensemble": {"f1":[],"em":[]}}
for batch in val_dsB.take(8):
  for i in range(31):
    input_ids = batch[0]["input_ids"][i]
    real_start, real_end = np.argmax(batch[1]["out_S"][i]), np.argmax(batch[1]["out_E"][i])
    answer = tokenizer.decode(input_ids[real_start:real_end+1])
    inputD = batch[0].copy()
    inputD.pop("token_type_ids")

    probability_B = modelB.predict(batch[0])
    predicted_limitsB = start_end_token_from_probabilities(*probability_B, 400)[i]
    predicted_answerB = tokenizer.decode(input_ids[predicted_limitsB[0]: predicted_limitsB[1]+1])
    probability_D = model.predict(inputD)
    predicted_limitsD = start_end_token_from_probabilities(*probability_D, 400)[i]
    predicted_answerD = tokenizer.decode(input_ids[predicted_limitsD[0]: predicted_limitsD[1]+1])

    probability_sum = [probability_B[0] +  probability_D[0], probability_B[1] +  probability_D[1]]
  
    predicted_limits_sum = start_end_token_from_probabilities(*probability_sum, 400)[i]
    predicted_answer_sum = tokenizer.decode(input_ids[predicted_limits_sum[0]: predicted_limits_sum[1]+1])

    scores["bert"]["f1"].append(compute_f1(answer, predicted_answerB))
    scores["bert"]["em"].append(compute_exact(answer, predicted_answerB))

    scores["distillbert"]["f1"].append(compute_f1(answer, predicted_answerD))
    scores["distillbert"]["em"].append(compute_exact(answer, predicted_answerD))

    scores["ensemble"]["f1"].append(compute_f1(answer, predicted_answer_sum))
    scores["ensemble"]["em"].append(compute_exact(answer, predicted_answer_sum))

    

In [ ]:

for key in scores.keys():
    total = len(scores[key]["f1"])
    print("Exact score {} for {}".format(100 * sum(scores[key]["em"]) / total, key))
    print("f1 score {} for {}".format(100 * sum(scores[key]["f1"]) / total, key))

Exact score 48.38709677419355 for bert
f1 score 68.3313570260737 for bert
Exact score 45.16129032258065 for distillbert
f1 score 66.2655556037091 for distillbert
Exact score 43.54838709677419 for ensemble
f1 score 67.7168112082046 for ensemble
